# AgentAuth Pocket CA Pipeline

Fine-tuning pipeline for Pocket CA on **Lambda Cloud GPU**.

### Quick Start

1. Run cells **in order** from top to bottom
2. Dependencies and NumPy fixes are handled automatically
3. Uses `NousResearch/Meta-Llama-3-8B-Instruct` (ungated mirror, no approval needed)
4. Wandb logging is **disabled by default** — set `USE_WANDB = True` to enable
5. Training takes ~2-4 hours on A100 with 80k samples

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

# Auto-detect project root
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    candidates = [
        PROJECT_ROOT / "agentauth-pocket-ca",
        Path.home() / "agentauth-pocket-ca",
    ]
    for parent in PROJECT_ROOT.parents:
        candidates.append(parent / "agentauth-pocket-ca")

    found = False
    for candidate in candidates:
        if (candidate / "configs").exists():
            PROJECT_ROOT = candidate.resolve()
            os.chdir(PROJECT_ROOT)
            found = True
            break

    if not found:
        raise RuntimeError(
            "Could not find the agentauth-pocket-ca project directory.\n"
            "Run: %cd ~/agentauth-pocket-ca"
        )
else:
    PROJECT_ROOT = PROJECT_ROOT.resolve()

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
# Fix NumPy version conflict (Lambda system torch is compiled against numpy 1.x)
# Then install training dependencies (vllm is skipped — only needed for serving)
%pip install -q "numpy<2" ipywidgets
%pip install -q accelerate bitsandbytes datasets fastapi peft pydantic pyyaml \
    sentencepiece transformers trl "uvicorn[standard]" wandb huggingface_hub

In [ ]:
# Verify environment
import torch
import transformers
import yaml

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} \u2014 {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if not torch.cuda.is_available():
    print("\n\u26a0\ufe0f  WARNING: No GPU detected. Training will be extremely slow.")

## HuggingFace Login

Enter your HuggingFace token below. Get one at https://huggingface.co/settings/tokens  
This is needed so subprocesses (training, eval) can download the model.

In [ ]:
# Set your HuggingFace token here
HF_TOKEN = ""  # <-- paste your token between the quotes

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token set for all subprocesses.")
else:
    # Try to get token from huggingface-cli login
    try:
        from huggingface_hub import get_token
        stored = get_token()
        if stored:
            os.environ["HF_TOKEN"] = stored
            os.environ["HUGGING_FACE_HUB_TOKEN"] = stored
            print("Using stored HF token.")
        else:
            print("No token found. Paste your token in HF_TOKEN above, or run:")
            print("  !huggingface-cli login")
    except Exception:
        print("No token found. Paste your token in HF_TOKEN above.")

In [ ]:
# Use ungated NousResearch mirror (no Meta approval needed)
# This patches model.yaml if it still points to the gated repo
model_yaml_path = PROJECT_ROOT / "configs/model.yaml"
model_config = yaml.safe_load(model_yaml_path.read_text())

if "meta-llama" in model_config.get("base_model_id", ""):
    model_config["base_model_id"] = "NousResearch/Meta-Llama-3-8B-Instruct"
    model_config["tokenizer_id"] = "NousResearch/Meta-Llama-3-8B-Instruct"
    model_yaml_path.write_text(yaml.dump(model_config, default_flow_style=False))
    print("Switched to ungated mirror: NousResearch/Meta-Llama-3-8B-Instruct")

training_config = yaml.safe_load((PROJECT_ROOT / "configs/training.yaml").read_text())

print("\nModel config:")
print(json.dumps(model_config, indent=2))
print("\nTraining config:")
print(json.dumps(training_config, indent=2))

## Lambda Run Settings

Set these values before launching the build or training cells.  
Leave the dataset paths empty if you only want synthetic data.

In [ ]:
DATASET_SIZE = 100000
RUN_NAME = "pocket-ca-v1"
FORCE_REBUILD_DATASET = True   # Always rebuild to ensure fresh full dataset
RESUME_FROM_CHECKPOINT = ""
USE_WANDB = False  # Set to True if you have a wandb account

FINQA_PATH = ""
CONVFINQA_PATH = ""
FINANCEBENCH_PATH = ""
FINR1_PATH = ""

# Disable wandb unless opted in
if not USE_WANDB:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"
else:
    os.environ.pop("WANDB_DISABLED", None)
    os.environ.pop("WANDB_MODE", None)


def run_command(command: list[str], extra_env: dict[str, str] | None = None) -> None:
    """Run a shell command with live stdout/stderr output."""
    env = os.environ.copy()
    if extra_env:
        env.update({k: v for k, v in extra_env.items() if v})
    cmd_str = " ".join(shlex.quote(p) for p in command)
    print(f"$ {cmd_str}")
    result = subprocess.run(
        command, cwd=PROJECT_ROOT, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {cmd_str}\n"
            f"{result.stdout}"
        )


dataset_env = {
    "FINQA_PATH": FINQA_PATH,
    "CONVFINQA_PATH": CONVFINQA_PATH,
    "FINANCEBENCH_PATH": FINANCEBENCH_PATH,
    "FINR1_PATH": FINR1_PATH,
}

print(json.dumps({
    "DATASET_SIZE": DATASET_SIZE,
    "RUN_NAME": RUN_NAME,
    "USE_WANDB": USE_WANDB,
    "FORCE_REBUILD_DATASET": FORCE_REBUILD_DATASET,
    "RESUME_FROM_CHECKPOINT": RESUME_FROM_CHECKPOINT or None,
}, indent=2))

## Build Dataset

Generates synthetic financial reasoning samples and prepares train/validation/test splits.  
If you set any dataset paths above, those are merged in during the build.

In [ ]:
build_env = dict(dataset_env)
build_env["SKIP_TOKENIZER_VALIDATION"] = "1"

run_command(["bash", "scripts/build_dataset.sh", str(DATASET_SIZE)], extra_env=build_env)

In [ ]:
# Verify dataset was built correctly
train_path = PROJECT_ROOT / "data/training/train.jsonl"
val_path = PROJECT_ROOT / "data/training/validation.jsonl"
test_path = PROJECT_ROOT / "data/training/test.jsonl"

train_count = sum(1 for _ in train_path.open())
val_count = sum(1 for _ in val_path.open())
test_count = sum(1 for _ in test_path.open())

batch_size = training_config["per_device_train_batch_size"]
grad_accum = training_config["gradient_accumulation_steps"]
epochs = training_config["num_train_epochs"]
steps_per_epoch = train_count // (batch_size * grad_accum)

print(f"Train:      {train_count:,}")
print(f"Validation: {val_count:,}")
print(f"Test:       {test_count:,}")
print(f"Total:      {train_count + val_count + test_count:,}")
print(f"\nEffective batch size: {batch_size} x {grad_accum} = {batch_size * grad_accum}")
print(f"Steps per epoch:  {steps_per_epoch:,}")
print(f"Total steps ({epochs} epochs): {steps_per_epoch * epochs:,}")

print("\nSample record:")
sample = json.loads(train_path.read_text().splitlines()[0])
print(json.dumps({k: sample[k] for k in ['id', 'category', 'instruction'] if k in sample}, indent=2))

## Train QLoRA Adapter

Fine-tunes Llama 3 8B with QLoRA (4-bit quantization + LoRA adapters).  
**This takes ~2-4 hours on a single A100.** Let it run.

Set `RESUME_FROM_CHECKPOINT` above to continue an interrupted run.

In [ ]:
train_env = dict(dataset_env)
train_env["SKIP_TOKENIZER_VALIDATION"] = "1"
# Pass HF token to subprocess
if os.environ.get("HF_TOKEN"):
    train_env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    train_env["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
# Don't rebuild dataset again inside launch_training.sh (already built above)
# But do rebuild if FORCE_REBUILD_DATASET is set
if FORCE_REBUILD_DATASET:
    train_env["FORCE_REBUILD_DATASET"] = "1"
if RESUME_FROM_CHECKPOINT:
    train_env["RESUME_FROM_CHECKPOINT"] = RESUME_FROM_CHECKPOINT

run_command(["bash", "scripts/launch_training.sh", RUN_NAME, str(DATASET_SIZE)], extra_env=train_env)

## Evaluate

Runs reasoning and budget benchmarks against the trained checkpoint.

In [ ]:
eval_env = {}
if os.environ.get("HF_TOKEN"):
    eval_env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    eval_env["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

checkpoint_path = PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME
if not checkpoint_path.exists():
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Run the training cell first, or set RUN_NAME to an existing checkpoint.")
else:
    print(f"Evaluating checkpoint: {checkpoint_path}")
    run_command(["python", "evaluation/eval_reasoning.py", "--checkpoint", str(checkpoint_path), "--limit", "100"], extra_env=eval_env)
    run_command(["python", "evaluation/eval_budget.py", "--checkpoint", str(checkpoint_path), "--limit", "100"], extra_env=eval_env)

## Serve the Model (Optional)

Starts a FastAPI inference server on port 8000.  
For vLLM serving (merged checkpoints only), first run `%pip install vllm`  
then set `POCKET_CA_USE_VLLM=1` before running.

In [ ]:
checkpoint_path = PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME
if not checkpoint_path.exists():
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Run the training cell first.")
else:
    os.environ["POCKET_CA_MODEL_PATH"] = str(checkpoint_path)
    print(f"POCKET_CA_MODEL_PATH = {checkpoint_path}")
    print("Starting server on http://0.0.0.0:8000 ...")
    run_command(["uvicorn", "serving.inference_api:app", "--host", "0.0.0.0", "--port", "8000"])